In [ ]:
from __future__ import annotations

from pathlib import Path
import base64
import html
import io
import json
import math
import os
import shutil
import subprocess
import sys
import threading
import time
import urllib.parse
import urllib.request
import zipfile

from fastapi import Request


WORK = Path(os.environ.get("COLAB_PUBLIC_SITE_WORK", "/content/colab_public_face_site"))
WORK.mkdir(parents=True, exist_ok=True)
URL_FILE = Path(os.environ.get("COLAB_PUBLIC_SITE_URL_FILE", "/content/colab_public_face_site_url.txt"))
PORT = int(os.environ.get("COLAB_PUBLIC_SITE_PORT", "7864"))
SHARE = os.environ.get("COLAB_PUBLIC_SITE_SHARE", "0") == "1"
BLOCK = os.environ.get("COLAB_PUBLIC_SITE_BLOCK", "1") != "0"
DEFAULT_MIN_SCORE = float(os.environ.get("COLAB_PUBLIC_SITE_DEFAULT_MIN_SCORE", "0.89"))
DEFAULT_MIN_MARGIN = float(os.environ.get("COLAB_PUBLIC_SITE_DEFAULT_MIN_MARGIN", "0.02"))
IMG_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".webp"}

detector = None
detector_lock = threading.Lock()
IDENTITIES = ("Adi", "Faraj", "Slava")
_HERO_DATA_URI = ""
FACE_IDENTITY_ENGINE_DIR = os.environ.get("FACE_IDENTITY_ENGINE_DIR", "FaceIdentityEngine")
HIVE_ASSET_BASE_URL = os.environ.get(
    "COLAB_PUBLIC_HIVE_ASSET_BASE_URL",
    "",
).rstrip("/")
URSINA_INSTALLER_URL = os.environ.get(
    "COLAB_PUBLIC_URSINA_INSTALLER_URL",
    "https://github.com/Holodininyaroslav/bee-face-recognition-project/releases/latest/download/bee_ursina_game_installer.zip",
).strip()
BEEBOARD_INSTALLER_URL = os.environ.get(
    "COLAB_PUBLIC_BEEBOARD_INSTALLER_URL",
    "https://github.com/Holodininyaroslav/bee-face-recognition-project/releases/latest/download/beeboard_interface_installer.zip",
).strip()
PHYSICAL_INSTALLER_URL = os.environ.get(
    "COLAB_PUBLIC_PHYSICAL_INSTALLER_URL",
    "https://github.com/Holodininyaroslav/bee-face-recognition-project/releases/latest/download/physical_simulation_installer.zip",
).strip()
HIVE_STATE_LOCK = threading.Lock()
HIVE_STATE: dict | None = None
LOCAL_URSINA_PROCESS: subprocess.Popen | None = None
LOCAL_AI_MIPS_PROCESS: subprocess.Popen | None = None
LOCAL_BEEBOARD_PROCESS: subprocess.Popen | None = None

LANGUAGES = {
    "Русский": "ru",
    "English": "en",
    "עברית": "he",
}

TEXT = {
    "ru": {
        "project": "Bee Face recognition project",
        "welcome": "Bee Face recognition project",
        "tagline": "Демонстрационный стенд распознавания лиц на базе Google Colab.",
        "intro": (
            "Здесь можно загрузить изображение или скриншот, отправить его в Colab-детектор "
            "и получить ответ по тем же принципам, по которым распознаватель подключается к проекту."
        ),
        "simple": "Простая демонстрация",
        "complex": "Сложная демонстрация, интегрированная в проект",
        "simple_title": "Простая демонстрация распознавания",
        "simple_note": "Загрузите картинку, выберите GPU или CPU и нажмите Recognize.",
        "complex_title": "Сложная демонстрация проекта",
        "complex_note": (
            "Это отдельная страница для будущей интегрированной демонстрации: здесь можно будет "
            "собрать сценарий игры, сетевой интерфейс и визуальную подачу результатов."
        ),
        "back": "Назад",
        "image": "Изображение / скриншот",
        "mode": "Режим вычислений",
        "score": "Минимальный score",
        "margin": "Минимальный margin",
        "recognize": "Recognize",
        "json": "JSON детектора",
        "summary": "Загрузите изображение и нажмите Recognize.",
        "examples": "Проверки на встроенных статуях",
        "language": "Язык",
        "status_gpu": "Colab GPU/CPU",
        "status_api": "Открытый upload API",
        "status_project": "Готово для интеграции",
    },
    "en": {
        "project": "Bee Face recognition project",
        "welcome": "Bee Face recognition project",
        "tagline": "A Google Colab powered face recognition demonstration.",
        "intro": (
            "Upload an image or screenshot, send it to the Colab detector, and receive a result "
            "using the same recognition flow that can be connected to the project interface."
        ),
        "simple": "Simple demonstration",
        "complex": "Complex demonstration integrated into the project",
        "simple_title": "Simple recognition demonstration",
        "simple_note": "Upload an image, choose GPU or CPU, and press Recognize.",
        "complex_title": "Complex project demonstration",
        "complex_note": (
            "This is a separate page for the future integrated demonstration. Later we can place "
            "the game scenario, network interface, and visual result stream here."
        ),
        "back": "Back",
        "image": "Image / screenshot",
        "mode": "Compute mode",
        "score": "Minimum score",
        "margin": "Minimum margin",
        "recognize": "Recognize",
        "json": "Detector JSON",
        "summary": "Upload an image and press Recognize.",
        "examples": "Built-in statue checks",
        "language": "Language",
        "status_gpu": "Colab GPU/CPU",
        "status_api": "Public upload API",
        "status_project": "Ready for integration",
    },
    "he": {
        "project": "Bee Face recognition project",
        "welcome": "Bee Face recognition project",
        "tagline": "הדגמת זיהוי פנים המבוססת על Google Colab.",
        "intro": (
            "אפשר להעלות תמונה או צילום מסך, לשלוח אותם לגלאי ב-Colab ולקבל תשובה "
            "באותו עקרון שישמש לחיבור המזהה לממשק הפרויקט."
        ),
        "simple": "הדגמה פשוטה",
        "complex": "הדגמה מורכבת המשולבת בפרויקט",
        "simple_title": "הדגמת זיהוי פשוטה",
        "simple_note": "העלו תמונה, בחרו GPU או CPU ולחצו Recognize.",
        "complex_title": "הדגמה מורכבת של הפרויקט",
        "complex_note": (
            "זהו עמוד נפרד להדגמה המשולבת העתידית. בהמשך נוכל להוסיף כאן את תרחיש המשחק, "
            "ממשק הרשת והצגת תוצאות חזותית."
        ),
        "back": "חזרה",
        "image": "תמונה / צילום מסך",
        "mode": "מצב חישוב",
        "score": "סף score מינימלי",
        "margin": "סף margin מינימלי",
        "recognize": "Recognize",
        "json": "JSON של הגלאי",
        "summary": "העלו תמונה ולחצו Recognize.",
        "examples": "בדיקות פסלים מובנות",
        "language": "שפה",
        "status_gpu": "Colab GPU/CPU",
        "status_api": "API העלאה ציבורי",
        "status_project": "מוכן לאינטגרציה",
    },
}


def _payload_candidates() -> list[Path]:
    here = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
    raw = [
        os.environ.get("COLAB_PUBLIC_SITE_PAYLOAD", ""),
        "/content/colab_ai_mips_bee_identity_payload.zip",
        "/content/colab_cuda_payload.zip",
        str(here / "colab_ai_mips_bee_identity_payload.zip"),
        str(here / "Face_detector" / "colab_cuda_payload.zip"),
    ]
    return [Path(value) for value in raw if value]


def _looks_extracted() -> bool:
    return (
        (WORK / "colab_ai_mips_bee_world.py").exists()
        and (WORK / "models" / "deepid_weights.bin").exists()
    )


def _extract_payload(path: Path) -> bool:
    if not path.exists():
        return False
    with zipfile.ZipFile(path, "r") as zf:
        zf.extractall(WORK)
    print("Extracted payload:", path, "->", WORK, flush=True)
    return _looks_extracted()


def _import_gradio():
    try:
        import gradio as gr
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gradio"])
        import gradio as gr
    return gr


class LocalExeDetector:
    def __init__(self, project_root: Path):
        self.project_root = project_root
        self.face_root = project_root / "Face_detector"
        self.references = self.face_root / "references"
        self.engine_root = _identity_engine_root(project_root)
        self.matcher = self.engine_root / "identity_matcher.py"
        self.python = self.engine_root / ".venv" / "Scripts" / "python.exe"
        if not self.python.exists():
            self.python = Path(sys.executable)
        self.min_score = 0.89
        self.min_margin = 0.04
        self.ref_items = [(label, p) for label in IDENTITIES for p in sorted((self.references / label).glob("*.png"))]

    def load_references(self, mode: str = "gpu") -> None:
        return None

    def _detector_exe(self, mode: str) -> Path:
        if mode.lower() == "gpu":
            return self.face_root / "build_opencl" / "Release" / "face_detector.exe"
        return self.face_root / "build_cpu" / "Release" / "face_detector.exe"

    def detect_image(self, image_path: str | Path, mode: str = "gpu") -> dict:
        output_dir = self.project_root / "_colab_public_site_local_test_output"
        command = [
            str(self.python),
            str(self.matcher),
            "--image",
            str(image_path),
            "--bee-id",
            "0",
            "--faces-root",
            str(self.face_root),
            "--references",
            str(self.references),
            "--output-dir",
            str(output_dir),
            "--detector-exe",
            str(self._detector_exe(mode)),
            "--backend",
            mode.lower(),
            "--labels",
            ",".join(IDENTITIES),
            "--min-score",
            str(self.min_score),
            "--min-margin",
            str(self.min_margin),
        ]
        process = subprocess.run(command, text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, timeout=180)
        text = process.stdout.strip() or process.stderr.strip()
        result = json.loads(text) if text else {}
        result.setdefault("returncode", process.returncode)
        result.setdefault("elapsed_ms", result.get("total_ms", 0.0))
        return result


def _identity_engine_root(project_root: Path) -> Path:
    preferred = project_root / FACE_IDENTITY_ENGINE_DIR
    if (preferred / "identity_matcher.py").exists() or (preferred / "main.py").exists():
        return preferred
    candidates: list[tuple[int, Path]] = []
    try:
        for candidate in sorted(project_root.iterdir(), key=lambda p: p.name.lower()):
            if not candidate.is_dir():
                continue
            lowered = candidate.name.lower()
            if lowered.startswith(("backup", "_", ".")):
                continue
            has_matcher = (candidate / "identity_matcher.py").exists()
            has_main = (candidate / "main.py").exists()
            if has_matcher or has_main:
                score = (4 if has_main else 0) + (2 if has_matcher else 0) + (1 if (candidate / ".venv").exists() else 0)
                candidates.append((score, candidate))
    except OSError:
        pass
    if candidates:
        return max(candidates, key=lambda item: (item[0], item[1].name.lower()))[1]
    return preferred


def _local_project_root() -> Path | None:
    if "__file__" not in globals():
        return None
    root = Path(__file__).resolve().parent
    if (_identity_engine_root(root) / "identity_matcher.py").exists() and (root / "Face_detector" / "references").exists():
        return root
    return None


def ensure_payload() -> None:
    if _looks_extracted():
        return
    for candidate in _payload_candidates():
        if _extract_payload(candidate):
            return
    try:
        from google.colab import files  # type: ignore
    except Exception as exc:
        raise FileNotFoundError(
            "Detector payload was not found. Put colab_ai_mips_bee_identity_payload.zip "
            "next to this script or upload it in Colab."
        ) from exc
    print("Upload colab_ai_mips_bee_identity_payload.zip", flush=True)
    uploaded = files.upload()
    for name, data in uploaded.items():
        if not name.lower().endswith(".zip"):
            continue
        path = Path("/content") / name
        path.write_bytes(data)
        if _extract_payload(path):
            return
    raise FileNotFoundError("Uploaded ZIP did not contain the Colab face detector payload.")


def load_detector():
    global detector, IDENTITIES
    ensure_payload()
    if str(WORK) not in sys.path:
        sys.path.insert(0, str(WORK))
    from colab_ai_mips_bee_world import DeepIDIdentityDetector, IDENTITIES as module_identities

    IDENTITIES = tuple(module_identities)
    if detector is None:
        detector = DeepIDIdentityDetector(WORK)
        try:
            detector.load_references("gpu")
        except Exception as exc:
            project_root = _local_project_root()
            if project_root is None:
                raise
            print("PyTorch detector is not available locally; using face_detector.exe fallback for local checks:", exc, flush=True)
            detector = LocalExeDetector(project_root)
        detector.min_score = DEFAULT_MIN_SCORE
        detector.min_margin = DEFAULT_MIN_MARGIN
        print("References loaded:", len(detector.ref_items), flush=True)
    return detector


def _image_paths(folder: Path) -> list[Path]:
    if not folder.exists():
        return []
    return sorted([p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in IMG_EXTS])


def example_images() -> list[list[str]]:
    examples: list[list[str]] = []
    for label in IDENTITIES:
        folders = [
            WORK / "identity_test_photos" / label,
            WORK / "identity_references" / label,
            WORK / "Face_detector" / "references" / label,
        ]
        image = None
        for folder in folders:
            paths = _image_paths(folder)
            if paths:
                image = paths[0]
                break
        if image is not None:
            examples.append([str(image), "GPU", float(detector.min_score), float(detector.min_margin)])
    return examples


def _summary(payload: dict) -> str:
    if not payload.get("ok"):
        return f"### Error\n\n`{payload.get('error', 'unknown error')}`"
    status = "ACCEPTED" if payload.get("accepted") else "UNKNOWN"
    return (
        f"### {status}: {payload.get('identity', 'Unknown')}\n\n"
        f"| field | value |\n"
        f"|---|---:|\n"
        f"| mode | {payload.get('mode', '')} |\n"
        f"| best label | {payload.get('best_label', 'Unknown')} |\n"
        f"| best score | {float(payload.get('best_score', 0.0) or 0.0):.6f} |\n"
        f"| runner up | {payload.get('runner_up_label', 'Unknown')} |\n"
        f"| margin | {float(payload.get('margin', 0.0) or 0.0):.6f} |\n"
        f"| elapsed | {float(payload.get('elapsed_ms', 0.0) or 0.0):.1f} ms |\n"
    )


def detect_uploaded(image_path, raw_mode: str = "GPU", min_score: float | None = None, min_margin: float | None = None):
    active = load_detector()
    if not image_path:
        payload = {"ok": False, "accepted": False, "identity": "Unknown", "error": "Upload an image first."}
        return _summary(payload), payload
    mode = "gpu" if str(raw_mode or "GPU").upper() == "GPU" else "cpu"
    started = time.perf_counter()
    with detector_lock:
        old_min_score = float(active.min_score)
        old_min_margin = float(active.min_margin)
        try:
            if min_score is not None:
                active.min_score = float(min_score)
            if min_margin is not None:
                active.min_margin = float(min_margin)
            active.load_references(mode)
            result = active.detect_image(Path(image_path), mode=mode)
        finally:
            active.min_score = old_min_score
            active.min_margin = old_min_margin
    elapsed_ms = float(result.get("elapsed_ms", (time.perf_counter() - started) * 1000.0))
    accepted = bool(result.get("accepted"))
    best_label = str(result.get("best_label") or result.get("identity") or "Unknown")
    payload = {
        "ok": True,
        "backend": "colab-public-face-site",
        "mode": str(raw_mode or "GPU").upper(),
        "accepted": accepted,
        "identity": best_label if accepted else "Unknown",
        "best_label": best_label,
        "best_score": float(result.get("best_score", 0.0) or 0.0),
        "runner_up_label": result.get("runner_up_label", "Unknown"),
        "runner_up_score": result.get("runner_up_score", 0.0),
        "margin": result.get("margin", 0.0),
        "best_variant": result.get("best_variant", ""),
        "matched_reference": result.get("matched_reference", ""),
        "elapsed_ms": elapsed_ms,
        "image": str(image_path),
        "thresholds": {
            "min_score": float(active.min_score if min_score is None else min_score),
            "min_margin": float(active.min_margin if min_margin is None else min_margin),
        },
        "raw": result,
    }
    return _summary(payload), payload


def _lang_code(language: str | None) -> str:
    return LANGUAGES.get(str(language or "Русский"), "ru")


def _text(language: str | None) -> dict:
    return TEXT[_lang_code(language)]


def _direction(language: str | None) -> str:
    return "rtl" if _lang_code(language) == "he" else "ltr"


def _hero_data_uri() -> str:
    global _HERO_DATA_URI
    if _HERO_DATA_URI:
        return _HERO_DATA_URI
    try:
        from PIL import Image, ImageDraw

        width, height = 1280, 560
        image = Image.new("RGB", (width, height), "#07101f")
        draw = ImageDraw.Draw(image, "RGBA")
        for y in range(height):
            r = int(8 + 18 * y / height)
            g = int(14 + 28 * y / height)
            b = int(31 + 34 * y / height)
            draw.line((0, y, width, y), fill=(r, g, b, 255))
        for x in range(-120, width, 120):
            draw.line((x, 0, x + 380, height), fill=(31, 164, 199, 34), width=2)
        for i in range(44):
            angle = i * 0.45
            cx = int(width * 0.68 + math.cos(angle) * (210 + i * 2))
            cy = int(height * 0.48 + math.sin(angle) * (115 + i))
            radius = 4 + (i % 5)
            color = (255, 190, 35, 155) if i % 3 == 0 else (76, 221, 182, 125)
            draw.ellipse((cx - radius, cy - radius, cx + radius, cy + radius), fill=color)
            if i > 0:
                px = int(width * 0.68 + math.cos((i - 1) * 0.45) * (210 + (i - 1) * 2))
                py = int(height * 0.48 + math.sin((i - 1) * 0.45) * (115 + (i - 1)))
                draw.line((px, py, cx, cy), fill=(80, 230, 190, 55), width=2)
        face_x, face_y = int(width * 0.69), int(height * 0.50)
        draw.rounded_rectangle((face_x - 150, face_y - 190, face_x + 150, face_y + 190), radius=95, outline=(114, 230, 210, 190), width=5)
        draw.arc((face_x - 72, face_y - 70, face_x - 14, face_y - 12), 190, 350, fill=(255, 198, 44, 210), width=5)
        draw.arc((face_x + 14, face_y - 70, face_x + 72, face_y - 12), 190, 350, fill=(255, 198, 44, 210), width=5)
        draw.arc((face_x - 70, face_y + 38, face_x + 70, face_y + 104), 15, 165, fill=(114, 230, 210, 180), width=5)
        for offset in (-265, -225, 225, 265):
            draw.line((face_x + offset, face_y - 200, face_x + offset, face_y + 200), fill=(255, 198, 44, 48), width=1)
        draw.rounded_rectangle((80, 76, 520, 474), radius=34, fill=(7, 16, 31, 138), outline=(255, 198, 44, 80), width=2)
        for row in range(6):
            y = 128 + row * 52
            draw.rounded_rectangle((120, y, 370 + row * 14, y + 20), radius=10, fill=(91, 208, 178, 82))
            draw.rectangle((395, y + 5, 470, y + 15), fill=(255, 198, 44, 105))
        buffer = io.BytesIO()
        image.save(buffer, format="PNG", optimize=True)
        _HERO_DATA_URI = "data:image/png;base64," + base64.b64encode(buffer.getvalue()).decode("ascii")
    except Exception:
        _HERO_DATA_URI = ""
    return _HERO_DATA_URI


def render_intro(language: str | None) -> str:
    t = _text(language)
    direction = _direction(language)
    hero = _hero_data_uri()
    image_style = f"background-image: linear-gradient(90deg, rgba(4,10,22,.94), rgba(4,10,22,.62), rgba(4,10,22,.25)), url('{hero}');" if hero else "background: linear-gradient(135deg, #08101f, #123344);"
    return f"""
    <section class="hero-shell" dir="{direction}" style="{image_style}">
      <div class="hero-copy">
        <div class="eyebrow">AI MIPS / Face Recognition</div>
        <h1>{t["welcome"]}</h1>
        <p class="tagline">{t["tagline"]}</p>
        <p class="intro-copy">{t["intro"]}</p>
        <div class="status-row">
          <span>{t["status_gpu"]}</span>
          <span>{t["status_api"]}</span>
          <span>{t["status_project"]}</span>
        </div>
      </div>
    </section>
    """


def render_simple_header(language: str | None) -> str:
    t = _text(language)
    direction = _direction(language)
    return f"""
    <section class="page-head" dir="{direction}">
      <div class="eyebrow">Simple mode</div>
      <h2>{t["simple_title"]}</h2>
      <p>{t["simple_note"]}</p>
    </section>
    """


def _complex_initial_state() -> dict:
    return {
        "processors": [
            {"id": "P0", "x": 50.0, "y": 58.0, "role": "worker", "group": "None", "manager": "none"}
        ],
        "selected": 0,
        "cycle": 0,
        "program": "Default",
        "events": [],
        "last": None,
    }


def _complex_selected(state: dict | None) -> dict:
    state = state or _complex_initial_state()
    processors = state.get("processors") or _complex_initial_state()["processors"]
    selected = max(0, min(int(state.get("selected", 0) or 0), len(processors) - 1))
    state["selected"] = selected
    return processors[selected]


def render_complex_topbar(language: str | None, state: dict | None) -> str:
    direction = _direction(language)
    state = state or _complex_initial_state()
    processors = state.get("processors") or []
    return f"""
    <section class="hive-copy hive-live" dir="{direction}">
      <header class="hive-topbar">
        <div>
          <div class="hive-eyebrow">AI MIPS / BEE SWARM CONTROL</div>
          <h2>AI MIPS Hive Service</h2>
        </div>
        <div class="hive-status">
          <span class="hive-pill ok">GPU/OpenCL: 2 platform</span>
          <span class="hive-pill ok">Colab: connected</span>
          <span class="hive-pill quiet">{len(processors)} processor{'s' if len(processors) != 1 else ''} | full mesh</span>
          <span class="hive-refresh">R</span>
        </div>
      </header>
    </section>
    """


def render_complex_map(state: dict | None) -> str:
    state = state or _complex_initial_state()
    processors = state.get("processors") or []
    nodes = []
    for index, proc in enumerate(processors):
        selected = " selected" if index == int(state.get("selected", 0) or 0) else ""
        role = html.escape(str(proc.get("role", "worker"))[:3].upper())
        nodes.append(
            f'<div class="hive-node{selected}" style="left:{float(proc.get("x", 50.0)):.1f}%;top:{float(proc.get("y", 58.0)):.1f}%">'
            f'<strong>{html.escape(str(proc.get("id", f"P{index}")))}</strong><small>{role}</small></div>'
        )
    links = []
    if len(processors) > 1:
        root = processors[0]
        for proc in processors[1:]:
            x1, y1 = float(root.get("x", 50.0)), float(root.get("y", 58.0))
            x2, y2 = float(proc.get("x", 50.0)), float(proc.get("y", 58.0))
            dx, dy = x2 - x1, y2 - y1
            length = math.sqrt(dx * dx + dy * dy)
            angle = math.degrees(math.atan2(dy, dx))
            links.append(
                f'<b class="hive-link" style="left:{x1:.1f}%;top:{y1:.1f}%;width:{length:.1f}%;transform:rotate({angle:.1f}deg)"></b>'
            )
    return f"""
    <div class="hive-map-live">
      <div class="hive-toolbar">
        <span class="hive-tool">-</span><span class="hive-tool">+</span><span class="hive-tool wide">Close menu</span>
        <span>mouse wheel: zoom | drag background: pan | click processor: actions</span>
      </div>
      <div class="hex-field">
        {''.join(f'<i style="--x:{x};--y:{y}"></i>' for y in range(7) for x in range(9))}
        {''.join(links)}
        {''.join(nodes)}
      </div>
    </div>
    """


def render_complex_details(state: dict | None) -> str:
    state = state or _complex_initial_state()
    proc = _complex_selected(state)
    last = state.get("last") or {}
    label = last.get("identity") or last.get("best_label") or "Unknown"
    elapsed = float(last.get("elapsed_ms", 0.0) or 0.0)
    mode = str(last.get("mode", "-")).upper()
    detected = f"{html.escape(str(label))} | {mode} | {elapsed:.0f} ms" if last else "Waiting for image"
    return f"""
    <div class="hive-card">
      <h3>Bee details</h3>
      <dl>
        <dt>Bee</dt><dd><strong>{html.escape(str(proc.get("id", "P0")))}</strong></dd>
        <dt>Role</dt><dd>{html.escape(str(proc.get("role", "worker")))}</dd>
        <dt>Group</dt><dd>{html.escape(str(proc.get("group", "None")))}</dd>
        <dt>Manager</dt><dd>{html.escape(str(proc.get("manager", "none")))}</dd>
        <dt>Cycle</dt><dd>{int(state.get("cycle", 0) or 0)}</dd>
        <dt>PC</dt><dd>0x{int(state.get("cycle", 0) or 0):08x}</dd>
        <dt>Inbox</dt><dd>{len(state.get("events") or [])}</dd>
        <dt>Detected</dt><dd>{detected}</dd>
        <dt>Position</dt><dd>{float(proc.get("x", 0.0)) * 12.6:.1f}, {float(proc.get("y", 0.0)) * 8.1:.1f}</dd>
      </dl>
    </div>
    """


def render_complex_detections(state: dict | None) -> str:
    state = state or _complex_initial_state()
    events = list(state.get("events") or [])
    rows = []
    for event in events[:12]:
        label = html.escape(str(event.get("identity") or event.get("best_label") or "Unknown"))
        bee = html.escape(str(event.get("bee", "P0")))
        mode = html.escape(str(event.get("mode", "-")).lower())
        elapsed = float(event.get("elapsed_ms", 0.0) or 0.0)
        accepted = "accepted" if event.get("accepted") else "unknown"
        rows.append(
            f'<article><div class="face-thumb"></div><strong>{bee}: {label}</strong><span>{elapsed:.0f} ms | {mode} | {accepted}</span></article>'
        )
    if not rows:
        rows.append('<article><div class="face-thumb"></div><strong>No detections yet</strong><span>Upload an image and run GPU or CPU</span></article>')
    return f"""
    <div class="hive-card detections-card">
      <div class="hive-section-title"><h3>Detections</h3><span>{len(events)} events</span></div>
      {''.join(rows)}
    </div>
    """


def _complex_outputs(language: str | None, state: dict | None):
    return (
        state,
        render_complex_topbar(language, state),
        render_complex_map(state),
        render_complex_details(state),
        render_complex_detections(state),
    )


def _complex_mutate(language: str | None, state: dict | None, action: str):
    state = json.loads(json.dumps(state or _complex_initial_state()))
    processors = state.setdefault("processors", [])
    if not processors:
        processors.extend(_complex_initial_state()["processors"])
    if action == "add":
        idx = len(processors)
        processors.append(
            {
                "id": f"P{idx}",
                "x": 42.0 + ((idx * 17) % 34),
                "y": 32.0 + ((idx * 23) % 38),
                "role": "worker",
                "group": "None",
                "manager": "P0" if idx else "none",
            }
        )
        state["selected"] = idx
    elif action == "reset":
        state = _complex_initial_state()
    elif action == "step":
        state["cycle"] = int(state.get("cycle", 0) or 0) + 1
    elif action == "hierarchy":
        for idx, proc in enumerate(processors):
            proc["manager"] = "none" if idx == 0 else "P0"
    elif action == "randomize":
        for idx, proc in enumerate(processors):
            proc["x"] = 22.0 + ((idx * 29 + int(time.time())) % 56)
            proc["y"] = 28.0 + ((idx * 19 + int(time.time())) % 48)
    elif action == "default":
        state["program"] = "Default"
    elif action == "controller":
        state["program"] = "Controller"
        processors[0]["role"] = "manager"
    return _complex_outputs(language, state)


def _complex_detect(image_path, state: dict | None, language: str | None, raw_mode: str, min_score: float, min_margin: float):
    state = json.loads(json.dumps(state or _complex_initial_state()))
    summary, payload = detect_uploaded(image_path, raw_mode, min_score, min_margin)
    state["cycle"] = int(state.get("cycle", 0) or 0) + 1
    proc = _complex_selected(state)
    event = dict(payload)
    event["bee"] = proc.get("id", "P0")
    state["last"] = event
    state.setdefault("events", []).insert(0, event)
    state["events"] = state["events"][:60]
    return (*_complex_outputs(language, state), summary, payload)


def _local_hive_asset(name: str) -> str:
    local = Path(__file__).resolve().parent / "python_ai_mips_sim" / "web" / name if "__file__" in globals() else Path()
    if local.exists():
        return local.read_text(encoding="utf-8")
    target = WORK / "hive_assets" / name
    target.parent.mkdir(parents=True, exist_ok=True)
    if not target.exists():
        if not HIVE_ASSET_BASE_URL:
            raise FileNotFoundError(
                f"Hive asset {name!r} is not bundled locally. Set COLAB_PUBLIC_HIVE_ASSET_BASE_URL only for an approved session."
            )
        urllib.request.urlretrieve(f"{HIVE_ASSET_BASE_URL}/{name}", target)
    return target.read_text(encoding="utf-8")


def _ursina_installer_path() -> Path:
    return _download_archive_path(
        ("bee_ursina_game_installer.zip", "bee-ursina-game-installer.zip"),
        URSINA_INSTALLER_URL,
        "bee_ursina_game_installer.zip",
    )


def _download_archive_path(names: tuple[str, ...], source_url: str, target_name: str) -> Path:
    for name in names:
        local = Path(__file__).resolve().parent / name if "__file__" in globals() else Path()
        if local.exists():
            return local
        work_file = WORK / name
        if work_file.exists():
            return work_file
    target = WORK / target_name
    if source_url and not target.exists():
        urllib.request.urlretrieve(source_url, target)
    return target


def _beeboard_installer_path() -> Path:
    return _download_archive_path(
        ("beeboard_interface_installer.zip", "beeboard-interface-installer.zip"),
        BEEBOARD_INSTALLER_URL,
        "beeboard_interface_installer.zip",
    )


def _physical_installer_path() -> Path:
    return _download_archive_path(
        ("physical_simulation_installer.zip", "physical-simulation-installer.zip"),
        PHYSICAL_INSTALLER_URL,
        "physical_simulation_installer.zip",
    )


def _write_local_ursina_control(processor_id: int) -> None:
    project_root = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
    with HIVE_STATE_LOCK:
        state = _hive_state()
        bee_count = max(len(state.get("nodes", [])), int(processor_id) + 1)
    bridge_path = project_root / "hive_bridge.json"
    control_path = project_root / "hive_control.json"
    data = {}
    if bridge_path.exists():
        try:
            loaded = json.loads(bridge_path.read_text(encoding="utf-8-sig"))
            if isinstance(loaded, dict):
                data = loaded
        except Exception:
            data = {}
    data["bee_count"] = max(int(bee_count), int(data.get("bee_count", 0) or 0))
    bridge_path.write_text(json.dumps(data, indent=2), encoding="utf-8")
    control_path.write_text(
        json.dumps(
            {
                "bee_count": int(bee_count),
                "control_id": int(processor_id),
                "request_time": time.time(),
            },
            indent=2,
        ),
        encoding="utf-8",
    )


def _start_local_ursina(api_url: str, processor_id: int) -> tuple[bool, str]:
    global LOCAL_URSINA_PROCESS
    project_root = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
    engine_root = _identity_engine_root(project_root)
    main_py = engine_root / "main.py"
    if not main_py.exists():
        return False, (
            "Local Ursina project was not found next to this server. "
            "Download the installer, extract it, run install_and_run.bat once, then run from the local Hive Web server."
        )
    _write_local_ursina_control(processor_id)
    if LOCAL_URSINA_PROCESS is not None and LOCAL_URSINA_PROCESS.poll() is None:
        return True, f"Local Ursina simulation is already running; control requested for P{processor_id}."
    python_exe = engine_root / ".venv" / "Scripts" / "pythonw.exe"
    if not python_exe.exists():
        python_exe = engine_root / ".venv" / "Scripts" / "python.exe"
    if not python_exe.exists():
        python_exe = Path(sys.executable)
    env = os.environ.copy()
    env["AI_MIPS_HIVE_API"] = api_url.rstrip("/")
    env["AI_MIPS_SELECTED_PROCESSOR"] = str(processor_id)
    log_path = project_root / "local_ursina_from_public_site.log"
    log_file = log_path.open("a", encoding="utf-8")
    LOCAL_URSINA_PROCESS = subprocess.Popen(
        [str(python_exe), str(main_py)],
        cwd=str(engine_root),
        env=env,
        stdout=log_file,
        stderr=subprocess.STDOUT,
    )
    return True, f"Local Ursina simulation started for P{processor_id}. Log: {log_path}"


def _url_responds(url: str, timeout: float = 1.5) -> bool:
    try:
        with urllib.request.urlopen(url, timeout=timeout) as response:
            return 200 <= response.status < 500
    except Exception:
        return False


def _process_creation_flags() -> int:
    return getattr(subprocess, "CREATE_NEW_PROCESS_GROUP", 0) | getattr(subprocess, "CREATE_NO_WINDOW", 0)


def _project_root() -> Path:
    return Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()


def _start_local_ai_mips_server() -> tuple[bool, str]:
    global LOCAL_AI_MIPS_PROCESS
    if _url_responds("http://127.0.0.1:8876/", timeout=0.8):
        return True, "AI MIPS Hive Web is already running on 127.0.0.1:8876."
    if LOCAL_AI_MIPS_PROCESS is not None and LOCAL_AI_MIPS_PROCESS.poll() is None:
        for _ in range(30):
            if _url_responds("http://127.0.0.1:8876/", timeout=0.5):
                return True, "AI MIPS Hive Web started on 127.0.0.1:8876."
            time.sleep(0.25)
        return False, "AI MIPS Hive Web process is running, but port 8876 is not responding yet."

    root = _project_root()
    app_dir = root / "python_ai_mips_sim"
    if not app_dir.exists():
        return False, f"AI MIPS server folder not found: {app_dir}"
    log_path = root / "server_8876_stdout.log"
    err_path = root / "server_8876_stderr.log"
    stdout_file = log_path.open("a", encoding="utf-8")
    stderr_file = err_path.open("a", encoding="utf-8")
    LOCAL_AI_MIPS_PROCESS = subprocess.Popen(
        [
            str(Path(sys.executable)),
            "-m",
            "ai_mips_sim.server",
            "--host",
            "127.0.0.1",
            "--port",
            "8876",
        ],
        cwd=str(app_dir),
        stdout=stdout_file,
        stderr=stderr_file,
        creationflags=_process_creation_flags(),
    )
    for _ in range(40):
        if _url_responds("http://127.0.0.1:8876/", timeout=0.5):
            return True, f"AI MIPS Hive Web started on 127.0.0.1:8876. Log: {log_path}"
        time.sleep(0.25)
    return False, f"AI MIPS Hive Web did not become ready. Logs: {log_path}, {err_path}"


def _beeboard_app_dir() -> Path:
    return Path.home() / "Desktop" / "Хомра" / "BeeBoard_v0_1" / "BeeBoard_Sim"


def _start_local_beeboard() -> tuple[bool, str]:
    global LOCAL_BEEBOARD_PROCESS
    if _url_responds("http://127.0.0.1:8877/api/health", timeout=0.8):
        return True, "BeeBoard is already running on 127.0.0.1:8877."
    if LOCAL_BEEBOARD_PROCESS is not None and LOCAL_BEEBOARD_PROCESS.poll() is None:
        for _ in range(30):
            if _url_responds("http://127.0.0.1:8877/api/health", timeout=0.5):
                return True, "BeeBoard started on 127.0.0.1:8877."
            time.sleep(0.25)
        return False, "BeeBoard process is running, but port 8877 is not responding yet."

    app_dir = _beeboard_app_dir()
    if not app_dir.exists():
        return False, f"BeeBoard service folder not found: {app_dir}"
    python_exe = app_dir / ".venv" / "Scripts" / "python.exe"
    if not python_exe.exists():
        python_exe = Path(sys.executable)
    log_path = app_dir / "beeboard_8877.log"
    log_file = log_path.open("a", encoding="utf-8")
    LOCAL_BEEBOARD_PROCESS = subprocess.Popen(
        [
            str(python_exe),
            "-m",
            "uvicorn",
            "app:app",
            "--host",
            "127.0.0.1",
            "--port",
            "8877",
        ],
        cwd=str(app_dir),
        stdout=log_file,
        stderr=subprocess.STDOUT,
        creationflags=_process_creation_flags(),
    )
    for _ in range(40):
        if _url_responds("http://127.0.0.1:8877/api/health", timeout=0.5):
            return True, f"BeeBoard started on 127.0.0.1:8877. Log: {log_path}"
        time.sleep(0.25)
    return False, f"BeeBoard did not become ready. Log: {log_path}"


def _hive_new_node(node_id: int, x: float, y: float, role: str = "worker") -> dict:
    return {
        "id": int(node_id),
        "role": role,
        "group_id": None,
        "manager_id": None,
        "cycle": 0,
        "pc": "0x00000000",
        "ready_inbox": 0,
        "detected": "",
        "detected_mode": "",
        "detected_ms": 0.0,
        "detected_time": "",
        "position": {"x": float(x), "y": float(y)},
        "bee_position": {"x": float(x), "y": float(y)},
        "code": ["LOAD R1, camera", "CALL face_detector", "STORE result"],
    }


def _hive_default_state() -> dict:
    return {
        "ok": True,
        "selected": 0,
        "link_mode": "full mesh",
        "nodes": [_hive_new_node(0, 630.0, 469.8)],
        "links": [],
        "bus": [],
        "detections": [],
        "events": [],
        "gpu": {"available": True, "platforms": 2},
        "detector": {
            "embedded_colab_enabled": True,
            "colab_url_configured": True,
            "colab_url_host": "connected",
            "raw_colab_url": "",
            "raw_colab_url_is_endpoint": True,
        },
        "board_modules": [
            {"name": "Camera", "body": "Frame source for the detector.", "kind": "sensor"},
            {"name": "Face detector", "body": "GPU/CPU recognition pipeline.", "kind": "compute"},
            {"name": "Hive bus", "body": "Events are routed between processors.", "kind": "network"},
        ],
        "soc_modules": [
            {"name": "BeeSoC", "body": "AI MIPS control core.", "kind": "root"},
            {"name": "Detector DMA", "body": "Image transfer.", "kind": "io"},
            {"name": "OpenCL/CUDA lane", "body": "Parallel recognizer.", "kind": "gpu"},
        ],
        "matrix_plan": None,
    }


def _hive_state() -> dict:
    global HIVE_STATE
    if HIVE_STATE is None:
        HIVE_STATE = _hive_default_state()
    return HIVE_STATE


def _hive_payload() -> dict:
    state = _hive_state()
    state["ok"] = True
    state["node_count"] = len(state.get("nodes", []))
    state["detector"]["colab_url_host"] = "connected"
    return state


def _hive_node(state: dict, node_id: int | None = None) -> dict:
    nodes = state.setdefault("nodes", [])
    if not nodes:
        nodes.append(_hive_new_node(0, 630.0, 469.8))
    selected = int(state.get("selected", 0) if node_id is None else node_id)
    for node in nodes:
        if int(node.get("id", 0)) == selected:
            state["selected"] = selected
            return node
    state["selected"] = int(nodes[0]["id"])
    return nodes[0]


def _hive_rebuild_links(state: dict, distance: float = 1.0, signal: float = 1.0) -> None:
    nodes = state.get("nodes", [])
    links = []
    for node in nodes[1:]:
        links.append(
            {
                "source": 0,
                "target": int(node["id"]),
                "distance": float(distance),
                "signal": float(signal),
                "kind": "manager-worker" if node.get("manager_id") is not None else "mesh",
            }
        )
    state["links"] = links


def _hive_event(state: dict, text: str) -> None:
    state.setdefault("bus", []).append({"source": 0, "target": int(state.get("selected", 0)), "payload": len(state.get("bus", [])) + 1})
    state.setdefault("events", []).append(text)
    state["bus"] = state["bus"][-32:]
    state["events"] = state["events"][-32:]


def _json_response(data: dict):
    from fastapi.responses import JSONResponse

    return JSONResponse(data)


def register_hive_web_routes(app) -> None:
    from fastapi import Request
    from fastapi.responses import HTMLResponse, PlainTextResponse

    @app.get("/hive-web")
    async def hive_web_page():
        html_doc = _local_hive_asset("index.html")
        html_doc = html_doc.replace('<link rel="stylesheet" href="/app.css?v=20260608-clean-canvas-hex-v2">', '<link rel="stylesheet" href="/hive-web/app.css">')
        html_doc = html_doc.replace('<script src="/app.js?v=20260609-single-bee-persist"></script>', '<script src="/hive-web/app.js"></script>')
        return HTMLResponse(html_doc)

    @app.get("/hive-web/app.css")
    async def hive_web_css():
        return PlainTextResponse(_local_hive_asset("app.css"), media_type="text/css")

    @app.get("/hive-web/app.js")
    async def hive_web_js():
        override = r'''

async function hiveRecognizeWithGradio(file, mode) {
  const uploadForm = new FormData();
  uploadForm.append("files", file, file.name || "manual.png");
  const uploadResponse = await fetch("/gradio_api/upload", { method: "POST", body: uploadForm });
  if (!uploadResponse.ok) throw new Error(`upload failed (${uploadResponse.status})`);
  const uploaded = await uploadResponse.json();
  const fileData = { path: uploaded[0], orig_name: file.name || "manual.png", meta: { _type: "gradio.FileData" } };
  const callResponse = await fetch("/gradio_api/call/recognize", {
    method: "POST",
    headers: { "content-type": "application/json" },
    body: JSON.stringify({ data: [fileData, mode, 0.89, 0.02] }),
  });
  if (!callResponse.ok) throw new Error(`recognize call failed (${callResponse.status})`);
  const call = await callResponse.json();
  const eventResponse = await fetch(`/gradio_api/call/recognize/${encodeURIComponent(call.event_id)}`);
  if (!eventResponse.ok) throw new Error(`recognize result failed (${eventResponse.status})`);
  const eventText = await eventResponse.text();
  const dataLine = eventText.split("\n").find((line) => line.startsWith("data: "));
  if (!dataLine) throw new Error("recognize returned no data");
  const result = JSON.parse(dataLine.slice(6));
  return Array.isArray(result) ? result[1] : result;
}

sendManualDetection = async function(mode) {
  const input = $("manualImageInput");
  const status = $("manualDetectStatus");
  const file = input?.files?.[0];
  if (!file) {
    if (status) status.textContent = "Choose an image first.";
    return;
  }
  const selected = Number(ui.selected ?? 0);
  if (status) status.textContent = `Sending to ${mode} detector through Colab...`;
  try {
    const payload = await hiveRecognizeWithGradio(file, mode);
    const identity = payload.identity || payload.best_label || "Unknown";
    const elapsed = Number(payload.elapsed_ms || 0);
    const recordUrl = `/hive-api/record-detection?processor_id=${encodeURIComponent(selected)}&identity=${encodeURIComponent(identity)}&mode=${encodeURIComponent(mode)}&elapsed_ms=${encodeURIComponent(elapsed)}&accepted=${encodeURIComponent(payload.accepted ? "1" : "0")}`;
    const record = await fetch(recordUrl);
    const data = await record.json();
    ui.hive = data.hive || ui.hive;
    ui.selected = Number(ui.hive?.selected ?? selected);
    renderAll();
    if (status) status.textContent = `${mode}: ${identity}, ${fmtMs(elapsed)}, backend: Colab Gradio API.`;
  } catch (err) {
    if (status) status.textContent = `Detector request failed: ${err.message || err}`;
  }
};
'''
        instant_menu_override = r'''

toggleMenuForNode = function(node) {
  const same = node.id === ui.selected;
  ui.selected = node.id;
  ui.menuOpen = same ? !ui.menuOpen : true;
  ui.controlVariantsOpen = false;
  ui.ursinaDownloadOpen = false;
  ui.physicalDownloadOpen = false;
  ui.boardDownloadOpen = false;
  ui.coreDownloadOpen = false;
  if (ui.menuOpen) focusNodeMenu(node);
  renderAll();
  apiPost(`/api/select?id=${encodeURIComponent(node.id)}`)
    .then((data) => {
      if (data?.ok) {
        ui.hive = data.hive || data;
        const refreshed = getNode(ui.selected);
        if (refreshed && ui.menuOpen) focusNodeMenu(refreshed);
        draw();
      }
    })
    .catch(() => {});
};

ui.coreDownloadOpen = Boolean(ui.coreDownloadOpen);

const colabEndpointForUi = `${window.location.origin}/hive-api/detect`;
const baseRenderBadgesForColabEndpoint = renderBadges;
renderBadges = function() {
  baseRenderBadgesForColabEndpoint();
  const colabInput = $("colabUrlInput");
  if (colabInput && document.activeElement !== colabInput && !colabInput.value.trim()) {
    colabInput.value = colabEndpointForUi;
  }
};

actionMenuCellsFor = function(point) {
  const cells = neighborCellsFor(point);
  const expanded = [...cells];
  const physicalCell = cells.find((cell) => cell.action === "physical");
  const boardCell = cells.find((cell) => cell.action === "board");
  const coreCell = cells.find((cell) => cell.action === "core");
  if (ui.physicalDownloadOpen && physicalCell) {
    const physicalChildren = hexNeighborCentersFor(physicalCell.center);
    expanded.push(
      { action: "physical_open", label: "Calibrate\nwings", center: physicalChildren.upperRight, variant: true },
      { action: "physical_download", label: "Download\ninstaller", center: physicalChildren.right, variant: true },
    );
  }
  if (ui.boardDownloadOpen && boardCell) {
    const boardChildren = hexNeighborCentersFor(boardCell.center);
    expanded.push(
      { action: "board_open", label: "Open\nBeeBoard", center: boardChildren.lowerRight, variant: true },
      { action: "board_download", label: "BeeBoard\ninstaller", center: boardChildren.right, variant: true },
    );
  }
  if (ui.coreDownloadOpen && coreCell) {
    const coreChildren = hexNeighborCentersFor(coreCell.center);
    expanded.push(
      { action: "core_open", label: "MIPS\nsimulator", center: coreChildren.upperRight, variant: true },
      { action: "core_download", label: "BeeBoard\ninstaller", center: coreChildren.right, variant: true },
    );
  }
  if (!ui.controlVariantsOpen) return expanded;
  const controlCell = cells.find((cell) => cell.action === "control");
  if (!controlCell) return expanded;
  const controlChildren = hexNeighborCentersFor(controlCell.center);
  const ursinaCell = { action: "physical_ursina", label: "Local Ursina\nsimulation", center: controlChildren.upperRight, variant: true };
  expanded.push(
    { action: "physical_browser", label: "Browser\nsimulation", center: controlChildren.upperLeft, variant: true },
    ursinaCell,
  );
  if (ui.ursinaDownloadOpen) {
    const downloadChildren = hexNeighborCentersFor(ursinaCell.center);
    expanded.push({ action: "physical_ursina_download", label: "Download\ninstaller", center: downloadChildren.right, variant: true });
  }
  return expanded;
};

const baseOpenActionForCoreBoard = openAction;
openAction = function(action) {
  const node = getSelectedNode();
  if (!node) return;
  if (action === "board") {
    ui.controlVariantsOpen = false;
    ui.ursinaDownloadOpen = false;
    ui.physicalDownloadOpen = false;
    ui.coreDownloadOpen = false;
    ui.boardDownloadOpen = !ui.boardDownloadOpen;
    ui.menuOpen = true;
    keepActionMenuInView(node);
    draw();
    return;
  }
  if (action === "core") {
    ui.controlVariantsOpen = false;
    ui.ursinaDownloadOpen = false;
    ui.physicalDownloadOpen = false;
    ui.boardDownloadOpen = false;
    ui.coreDownloadOpen = !ui.coreDownloadOpen;
    ui.menuOpen = true;
    keepActionMenuInView(node);
    draw();
    return;
  }
  if (action === "core_download") {
    window.location.href = "/downloads/beeboard-interface-installer.zip";
    return;
  }
  baseOpenActionForCoreBoard(action);
};
'''
        script = _local_hive_asset("app.js").replace("/api/detect?", "/hive-api/detect?") + override + instant_menu_override
        return PlainTextResponse(script, media_type="application/javascript")

    @app.get("/downloads/bee-ursina-game-installer.zip")
    async def download_ursina_installer():
        from fastapi.responses import FileResponse
        path = _ursina_installer_path()
        if not path.exists():
            return PlainTextResponse("Bee Ursina game installer archive was not found.", status_code=404)
        return FileResponse(path, media_type="application/zip", filename="bee_ursina_game_installer.zip")

    @app.get("/downloads/beeboard-interface-installer.zip")
    async def download_beeboard_installer():
        from fastapi.responses import FileResponse
        path = _beeboard_installer_path()
        if not path.exists():
            return PlainTextResponse("BeeBoard installer archive was not found.", status_code=404)
        return FileResponse(path, media_type="application/zip", filename="beeboard_interface_installer.zip")

    @app.get("/downloads/physical-simulation-installer.zip")
    async def download_physical_installer():
        from fastapi.responses import FileResponse
        path = _physical_installer_path()
        if not path.exists():
            return PlainTextResponse("Physical simulation installer archive was not found.", status_code=404)
        return FileResponse(path, media_type="application/zip", filename="physical_simulation_installer.zip")

    @app.get("/open-beeboard")
    async def open_beeboard(request: Request, target: str = "viewer", hive: str = "", processor: str = "0"):
        from fastapi.responses import HTMLResponse, RedirectResponse
        hash_target = "#processors" if target == "processors" else "#viewer"
        hive_url = hive.strip() or f"{request.url.scheme}://{request.url.netloc}/hive-web"
        params = urllib.parse.urlencode({"hive": hive_url, "processor": processor})
        local_url = f"http://127.0.0.1:8877/?{params}{hash_target}"
        ok, message = _start_local_beeboard()
        if ok:
            return RedirectResponse(local_url, status_code=302)
        body = (
            "<!doctype html><meta charset='utf-8'>"
            "<title>BeeBoard did not start</title>"
            "<body style='font-family:Segoe UI,Arial,sans-serif;background:#081018;color:#eef5ff;padding:24px'>"
            "<h1 style='color:#fca5a5'>BeeBoard did not start</h1>"
            f"<pre>{html.escape(message)}</pre>"
            f"<p>Target: <a style='color:#50c8ff' href='{html.escape(local_url)}'>{html.escape(local_url)}</a></p>"
            "</body>"
        )
        return HTMLResponse(body, status_code=503)

    @app.get("/physical-simulator")
    async def physical_simulator_page():
        from fastapi.responses import HTMLResponse
        body = """
<!doctype html>
<meta charset="utf-8">
<title>Physical wing calibration</title>
<style>
  body{margin:0;font-family:Segoe UI,Arial,sans-serif;background:#07101e;color:#eef5ff}
  main{padding:28px;max-width:980px;margin:auto}
  h1{color:#ffd052;margin:0 0 8px;font-size:36px}
  p{color:#bcd4ff;font-size:18px}
  canvas{display:block;width:100%;max-width:900px;height:360px;border:1px solid #33415f;background:#020611}
  button{margin-top:16px;padding:12px 22px;border:1px solid #415170;background:#162238;color:#fff;font-weight:800;font-size:16px}
</style>
<main>
  <div style="color:#32d5ff;font-weight:900;text-transform:uppercase">AI MIPS / physical wing simulator</div>
  <h1>Wing calibration simulation</h1>
  <p>Click the simulation or press Space to flap. This is the physical wing calibration view, not the browser 3D simulation.</p>
  <canvas id="wingCanvas" width="900" height="360" aria-label="Wing calibration simulation"></canvas>
  <button id="resetBtn">Reset calibration</button>
</main>
<script>
const canvas = document.getElementById("wingCanvas");
const ctx = canvas.getContext("2d");
const state = { y: 180, vy: -3, tick: 0, score: 0, wing: 0 };
function flap(){ state.vy = -8; state.wing = 1; }
function reset(){ state.y = 180; state.vy = -3; state.tick = 0; state.score = 0; state.wing = 0; }
document.getElementById("resetBtn").addEventListener("click", reset);
canvas.addEventListener("pointerdown", flap);
window.addEventListener("keydown", (event) => { if (event.code === "Space") { event.preventDefault(); flap(); } });
function draw(){
  state.tick += 1;
  state.vy += 0.34;
  state.y += state.vy;
  if (state.y > 318) { state.y = 318; state.vy = -5.4; state.score += 1; }
  if (state.y < 30) { state.y = 30; state.vy = 1.1; }
  state.wing *= 0.9;
  ctx.clearRect(0,0,900,360);
  ctx.fillStyle = "#07101e"; ctx.fillRect(0,0,900,360);
  ctx.strokeStyle = "rgba(70,95,135,.45)";
  for(let x=0;x<900;x+=45){ctx.beginPath();ctx.moveTo(x,0);ctx.lineTo(x,360);ctx.stroke();}
  for(let y=0;y<360;y+=45){ctx.beginPath();ctx.moveTo(0,y);ctx.lineTo(900,y);ctx.stroke();}
  ctx.fillStyle = "#10233d"; ctx.fillRect(660,40,84,112); ctx.fillRect(660,232,84,100);
  ctx.strokeStyle = "#38bdf8"; ctx.strokeRect(660,40,84,112); ctx.strokeRect(660,232,84,100);
  ctx.save();
  ctx.translate(220,state.y);
  ctx.rotate(Math.max(-0.55, Math.min(0.55, state.vy / 12)));
  ctx.fillStyle = "#fbbf24";
  ctx.beginPath(); ctx.ellipse(0,0,28,16,0,0,Math.PI*2); ctx.fill();
  ctx.fillStyle = "#111827"; ctx.fillRect(-16,-13,6,26); ctx.fillRect(0,-14,6,28); ctx.fillRect(16,-11,5,22);
  ctx.fillStyle = "rgba(237,245,255,.76)";
  ctx.beginPath(); ctx.ellipse(-9,-22,23,8,-0.5-state.wing,0,Math.PI*2); ctx.fill();
  ctx.beginPath(); ctx.ellipse(11,-22,23,8,0.5+state.wing,0,Math.PI*2); ctx.fill();
  ctx.restore();
  ctx.fillStyle = "#bcd4ff"; ctx.font = "18px Segoe UI, Arial";
  ctx.fillText(`calibration score: ${state.score}`, 24, 34);
  ctx.fillText(`altitude: ${Math.round(360 - state.y)}  velocity: ${state.vy.toFixed(1)}`, 24, 60);
  requestAnimationFrame(draw);
}
draw();
</script>
"""
        return HTMLResponse(body)

    @app.get("/local-ursina-simulator")
    async def local_ursina_simulator(request: Request, processor_id: str = "0", api: str = ""):
        try:
            selected = max(0, int(processor_id))
        except ValueError:
            selected = 0
        api_url = api.strip() or f"{request.url.scheme}://{request.url.netloc}"
        ok, message = _start_local_ursina(api_url, selected)
        color = "#86efac" if ok else "#fca5a5"
        status = "started" if ok else "could not start"
        body = (
            "<!doctype html><meta charset='utf-8'>"
            "<title>Local Ursina simulation</title>"
            "<body style='font-family:Segoe UI,Arial,sans-serif;background:#081018;color:#eef5ff;padding:24px'>"
            f"<h1 style='color:{color}'>Local Ursina simulation {status}</h1>"
            f"<p>Linked processor: P{selected}</p>"
            f"<pre>{html.escape(message)}</pre>"
            "</body>"
        )
        return HTMLResponse(body, status_code=200 if ok else 503)

    @app.get("/api/hive")
    async def api_hive():
        with HIVE_STATE_LOCK:
            return _json_response(_hive_payload())

    @app.get("/api/detector-status")
    async def api_detector_status():
        with HIVE_STATE_LOCK:
            return _json_response({"ok": True, "detector": _hive_payload()["detector"]})

    @app.post("/api/detector-config")
    async def api_detector_config(request: Request):
        body = await request.json()
        url = str(body.get("url", "")).strip()
        with HIVE_STATE_LOCK:
            state = _hive_state()
            state["detector"]["raw_colab_url"] = url
            state["detector"]["colab_url_configured"] = True
            state["detector"]["embedded_colab_enabled"] = True
            return _json_response({"ok": True, "hive": _hive_payload()})

    @app.post("/api/select")
    async def api_select(id: int = 0):
        with HIVE_STATE_LOCK:
            state = _hive_state()
            _hive_node(state, id)
            return _json_response({"ok": True, "hive": _hive_payload()})

    @app.post("/api/position")
    async def api_position(id: int = 0, x: float = 0.0, y: float = 0.0):
        with HIVE_STATE_LOCK:
            state = _hive_state()
            node = _hive_node(state, id)
            node["position"] = {"x": float(x), "y": float(y)}
            node["bee_position"] = {"x": float(x), "y": float(y)}
            return _json_response({"ok": True, "hive": _hive_payload()})

    @app.post("/api/add-processor")
    async def api_add_processor(distance: float = 1.0, signal: float = 1.0):
        with HIVE_STATE_LOCK:
            state = _hive_state()
            nodes = state.setdefault("nodes", [])
            node_id = max([int(node["id"]) for node in nodes] or [0]) + 1
            node = _hive_new_node(node_id, 630.0 + node_id * 95.0, 469.8 - (node_id % 3) * 78.0)
            node["manager_id"] = 0
            nodes.append(node)
            state["selected"] = node_id
            _hive_rebuild_links(state, distance, signal)
            _hive_event(state, f"Added processor P{node_id}")
            return _json_response({"ok": True, "hive": _hive_payload()})

    @app.post("/api/reset")
    async def api_reset(count: int = 1):
        global HIVE_STATE
        with HIVE_STATE_LOCK:
            HIVE_STATE = _hive_default_state()
            return _json_response({"ok": True, "hive": _hive_payload()})

    @app.post("/api/step-selected")
    async def api_step_selected():
        with HIVE_STATE_LOCK:
            state = _hive_state()
            node = _hive_node(state)
            node["cycle"] = int(node.get("cycle", 0)) + 1
            node["pc"] = f"0x{int(node['cycle']):08x}"
            _hive_event(state, f"Step P{node['id']}")
            return _json_response({"ok": True, "hive": _hive_payload()})

    @app.post("/api/run-selected")
    async def api_run_selected(cycles: int = 102):
        with HIVE_STATE_LOCK:
            state = _hive_state()
            node = _hive_node(state)
            node["cycle"] = int(node.get("cycle", 0)) + int(cycles)
            node["pc"] = f"0x{int(node['cycle']):08x}"
            _hive_event(state, f"Run P{node['id']} for {cycles} cycles")
            return _json_response({"ok": True, "hive": _hive_payload()})

    @app.post("/api/run-cluster-cpu")
    async def api_run_cluster_cpu(cycles: int = 102):
        with HIVE_STATE_LOCK:
            state = _hive_state()
            for node in state.get("nodes", []):
                node["cycle"] = int(node.get("cycle", 0)) + int(cycles)
                node["pc"] = f"0x{int(node['cycle']):08x}"
            _hive_event(state, f"All CPU run for {cycles} cycles")
            return _json_response({"ok": True, "hive": _hive_payload()})

    @app.post("/api/run-cluster-gpu")
    async def api_run_cluster_gpu(cycles: int = 102):
        with HIVE_STATE_LOCK:
            state = _hive_state()
            for node in state.get("nodes", []):
                node["cycle"] = int(node.get("cycle", 0)) + int(cycles)
                node["pc"] = f"0x{int(node['cycle']):08x}"
            _hive_event(state, f"All GPU run for {cycles} cycles")
            return _json_response({"ok": True, "hive": _hive_payload()})

    @app.post("/api/randomize-network")
    async def api_randomize_network():
        with HIVE_STATE_LOCK:
            state = _hive_state()
            for idx, node in enumerate(state.get("nodes", [])):
                node["position"] = {"x": 420.0 + ((idx * 137 + int(time.time())) % 520), "y": 260.0 + ((idx * 89 + int(time.time())) % 430)}
                node["bee_position"] = dict(node["position"])
            _hive_rebuild_links(state)
            return _json_response({"ok": True, "hive": _hive_payload()})

    @app.post("/api/build-hierarchy")
    async def api_build_hierarchy(managers: int = 2):
        with HIVE_STATE_LOCK:
            state = _hive_state()
            for idx, node in enumerate(state.get("nodes", [])):
                node["role"] = "manager" if idx < max(1, int(managers)) else "worker"
                node["manager_id"] = None if idx == 0 else 0
            _hive_rebuild_links(state)
            return _json_response({"ok": True, "hive": _hive_payload()})

    @app.post("/api/bus-send")
    async def api_bus_send(src: int = 0, dst: int = 0, payload: str = "0x1234"):
        with HIVE_STATE_LOCK:
            state = _hive_state()
            value = int(str(payload), 0) if str(payload).strip() else 0
            state.setdefault("bus", []).append({"source": int(src), "target": int(dst), "payload": value})
            return _json_response({"ok": True, "hive": _hive_payload()})

    @app.post("/api/matrix-plan")
    async def api_matrix_plan(request: Request):
        with HIVE_STATE_LOCK:
            state = _hive_state()
            state["matrix_plan"] = {"root": 0, "rows": 2, "cols": 2, "tasks": [{"processor": node["id"]} for node in state.get("nodes", [])]}
            return _json_response({"ok": True, "hive": _hive_payload()})

    @app.post("/api/program")
    async def api_program(kind: str = "default"):
        with HIVE_STATE_LOCK:
            state = _hive_state()
            node = _hive_node(state)
            node["code"] = [f"PROGRAM {kind}", "LOAD frame", "CALL detector", "STORE hive_event"]
            return _json_response({"ok": True, "hive": _hive_payload()})

    @app.post("/hive-api/detect")
    async def api_detect(request: Request, mode: str = "GPU", processor_id: int = 0, source: str = "manual-web"):
        suffix = ".png"
        ctype = request.headers.get("content-type", "")
        if "jpeg" in ctype or "jpg" in ctype:
            suffix = ".jpg"
        elif "webp" in ctype:
            suffix = ".webp"
        data = await request.body()
        upload = WORK / "hive_uploads" / f"manual_{time.time_ns()}{suffix}"
        upload.parent.mkdir(parents=True, exist_ok=True)
        upload.write_bytes(data)
        _summary_text, payload = detect_uploaded(str(upload), mode, DEFAULT_MIN_SCORE, DEFAULT_MIN_MARGIN)
        with HIVE_STATE_LOCK:
            state = _hive_state()
            node = _hive_node(state, int(processor_id))
            node["detected"] = payload.get("identity") or payload.get("best_label") or "Unknown"
            node["detected_mode"] = str(mode).upper()
            node["detected_ms"] = float(payload.get("elapsed_ms", 0.0) or 0.0)
            node["detected_time"] = time.strftime("%H:%M:%S")
            event = {
                "processor_id": int(node["id"]),
                "identity": node["detected"],
                "mode": str(mode).upper(),
                "elapsed_ms": node["detected_ms"],
                "timestamp_iso": time.strftime("%Y-%m-%dT%H:%M:%S"),
                "accepted": bool(payload.get("accepted")),
            }
            state.setdefault("detections", []).insert(0, event)
            state["detections"] = state["detections"][:60]
            result = dict(payload)
            result.update({"ok": True, "hive": _hive_payload(), "detected": node["detected"], "backend": "colab-hive-web"})
            return _json_response(result)

    @app.get("/hive-api/record-detection")
    async def hive_record_detection(processor_id: int = 0, identity: str = "Unknown", mode: str = "GPU", elapsed_ms: float = 0.0, accepted: str = "1"):
        with HIVE_STATE_LOCK:
            state = _hive_state()
            node = _hive_node(state, int(processor_id))
            node["detected"] = str(identity or "Unknown")
            node["detected_mode"] = str(mode).upper()
            node["detected_ms"] = float(elapsed_ms or 0.0)
            node["detected_time"] = time.strftime("%H:%M:%S")
            event = {
                "processor_id": int(node["id"]),
                "identity": node["detected"],
                "mode": node["detected_mode"],
                "elapsed_ms": node["detected_ms"],
                "timestamp_iso": time.strftime("%Y-%m-%dT%H:%M:%S"),
                "accepted": str(accepted).lower() not in {"0", "false", "no"},
            }
            state.setdefault("detections", []).insert(0, event)
            state["detections"] = state["detections"][:60]
            return _json_response({"ok": True, "hive": _hive_payload()})


def build_demo():
    gr = _import_gradio()
    active = load_detector()
    initial_language = "English"
    initial_text = _text(initial_language)

    def navigate(language: str | None, page: str):
        t = _text(language)
        return (
            page,
            gr.update(visible=page == "landing"),
            gr.update(visible=page == "simple"),
            gr.update(visible=page == "complex"),
            render_intro(language),
            render_simple_header(language),
            render_complex_topbar(language, _complex_initial_state()),
            gr.update(value=t["simple"]),
            gr.update(value=t["complex"]),
            gr.update(value=t["back"]),
            gr.update(value=t["back"]),
            gr.update(label=t["image"]),
            gr.update(label=t["mode"]),
            gr.update(label=t["score"]),
            gr.update(label=t["margin"]),
            gr.update(value=t["recognize"]),
            gr.update(label=t["json"]),
            t["summary"],
        )

    with gr.Blocks(
        title="Bee Face recognition project",
        css="""
        body, .gradio-container { background: #07101f !important; color: #f4f7fb !important; }
        .wrap { max-width: min(1680px, calc(100vw - 32px)); margin: 0 auto; }
        .wrap > .form, .wrap > div { border-radius: 8px !important; }
        .lang-select { max-width: 250px; margin: 0 0 18px auto; }
        .lang-select label { color: #dbeafe !important; }
        .hero-shell {
            min-height: 500px;
            border: 1px solid rgba(123, 176, 223, .22);
            border-radius: 8px;
            background-size: cover;
            background-position: center;
            display: flex;
            align-items: center;
            padding: 56px;
            overflow: hidden;
            box-shadow: 0 28px 80px rgba(0,0,0,.32);
        }
        .hero-copy { max-width: 720px; }
        .eyebrow {
            color: #72e5d3;
            font-weight: 800;
            letter-spacing: 0;
            text-transform: uppercase;
            font-size: 13px;
            margin-bottom: 10px;
        }
        .hero-copy h1 {
            font-size: 52px;
            line-height: 1.08;
            margin: 0 0 14px 0;
            color: #ffd54d;
        }
        .tagline {
            font-size: 21px;
            color: #f7fbff;
            margin: 0 0 14px 0;
        }
        .intro-copy {
            max-width: 660px;
            color: #b7c6dc;
            font-size: 16px;
            line-height: 1.55;
            margin: 0 0 24px 0;
        }
        .status-row {
            display: flex;
            flex-wrap: wrap;
            gap: 10px;
        }
        .status-row span {
            border: 1px solid rgba(114, 229, 211, .34);
            background: rgba(8, 18, 36, .7);
            color: #d9fff8;
            padding: 9px 12px;
            border-radius: 6px;
            font-weight: 700;
            font-size: 13px;
        }
        .choice-row button, .back-row button { min-height: 56px; font-weight: 800 !important; border-radius: 6px !important; }
        .choice-row button:first-child { background: #ffb000 !important; color: #07101f !important; border-color: #ffd54d !important; }
        .page-head, .complex-page {
            border: 1px solid rgba(123, 176, 223, .22);
            border-radius: 8px;
            padding: 26px 28px;
            background: linear-gradient(135deg, rgba(13, 29, 55, .92), rgba(7, 16, 31, .96));
            margin-bottom: 18px;
        }
        .page-head h2, .complex-page h2 {
            margin: 0 0 10px 0;
            color: #ffd54d;
            font-size: 28px;
        }
        .page-head p, .complex-page p {
            margin: 0;
            color: #b7c6dc;
            font-size: 15px;
            line-height: 1.55;
        }
        .complex-grid {
            display: grid;
            grid-template-columns: repeat(3, minmax(0, 1fr));
            gap: 12px;
            margin-top: 24px;
        }
        .complex-grid div {
            min-height: 112px;
            border: 1px solid rgba(255, 213, 77, .24);
            border-radius: 8px;
            padding: 18px;
            background: rgba(255, 255, 255, .04);
        }
        .complex-grid strong {
            display: block;
            color: #72e5d3;
            font-size: 24px;
            margin-bottom: 12px;
        }
        .complex-grid span { color: #f4f7fb; font-weight: 700; }
        .hive-copy {
            --h-bg: #090d16;
            --h-panel: #101827;
            --h-panel-2: #162237;
            --h-line: rgba(141,160,190,.22);
            --h-text: #edf5ff;
            --h-muted: #9fb0c9;
            --h-honey: #ffb000;
            --h-honey-2: #ffd057;
            --h-green: #22c55e;
            --h-cyan: #38bdf8;
            height: min(78vh, 820px);
            min-height: 680px;
            overflow: hidden;
            color: var(--h-text);
            border: 1px solid var(--h-line);
            border-radius: 8px;
            background: linear-gradient(135deg, #070b13 0%, #0b1020 55%, #0a111d 100%);
            font-family: "Segoe UI", Arial, sans-serif;
        }
        .hive-topbar {
            height: 74px;
            display: flex;
            align-items: center;
            justify-content: space-between;
            padding: 12px 18px;
            border-bottom: 1px solid var(--h-line);
            background: rgba(9, 13, 22, .92);
        }
        .hive-eyebrow { color: var(--h-cyan); font-size: 12px; font-weight: 800; text-transform: uppercase; }
        .hive-topbar h2 { color: var(--h-honey-2); font-size: 28px; margin: 0; }
        .hive-status { display: flex; align-items: center; gap: 10px; }
        .hive-pill {
            min-height: 34px;
            display: inline-flex;
            align-items: center;
            padding: 0 12px;
            border: 1px solid rgba(159,176,201,.25);
            background: rgba(159,176,201,.06);
            color: var(--h-muted);
            font-weight: 800;
            white-space: nowrap;
        }
        .hive-pill.ok { border-color: rgba(34,197,94,.45); color: #86efac; background: rgba(34,197,94,.08); }
        .hive-refresh { width: 38px; height: 36px; display: grid; place-items: center; border: 1px solid #40506d; background: #172237; font-weight: 900; }
        .hive-grid {
            height: calc(100% - 74px);
            display: grid;
            grid-template-columns: 320px minmax(0, 1fr) 360px;
        }
        .hive-left, .hive-right { min-height: 0; overflow: hidden; padding: 14px; background: rgba(11,17,29,.88); }
        .hive-left { border-right: 1px solid var(--h-line); }
        .hive-right { border-left: 1px solid var(--h-line); }
        .hive-panel, .hive-card {
            margin-bottom: 14px;
            padding: 14px;
            border: 1px solid var(--h-line);
            background: rgba(16,24,39,.82);
            box-shadow: 0 16px 36px rgba(0,0,0,.34);
        }
        .hive-panel h3, .hive-card h3 { color: var(--h-honey-2); margin: 0 0 12px; font-size: 18px; }
        .hive-panel label { display: inline-block; width: 86px; color: var(--h-muted); font-size: 13px; font-weight: 800; margin: 8px 0; }
        .hive-panel input {
            width: calc(100% - 92px);
            min-height: 44px;
            padding: 0 12px;
            border: 1px solid #40506d;
            color: var(--h-text);
            background: #0b1220;
            font-size: 18px;
        }
        .hive-buttons { display: grid; grid-template-columns: 1fr 1fr; gap: 8px; margin-top: 10px; }
        .hive-buttons button, .hive-toolbar button {
            min-height: 44px;
            border: 1px solid #40506d;
            color: var(--h-text);
            background: #172237;
            font-weight: 900;
            font-size: 16px;
        }
        .hive-buttons .hive-accent { background: var(--h-honey); color: #111827; border-color: var(--h-honey-2); }
        .hive-map { position: relative; overflow: hidden; background: #080d1b; }
        .hive-toolbar {
            position: absolute;
            z-index: 2;
            left: 12px; right: 12px; top: 12px;
            display: flex;
            gap: 8px;
            align-items: center;
            min-height: 54px;
            padding: 7px;
            border: 1px solid var(--h-line);
            background: rgba(10,16,28,.78);
            color: var(--h-muted);
            font-size: 18px;
        }
        .hive-toolbar button { min-width: 46px; padding: 0 12px; }
        .hex-field { position: absolute; inset: 0; overflow: hidden; }
        .hex-field i {
            position: absolute;
            left: calc(var(--x) * 118px - 40px);
            top: calc(var(--y) * 82px + 60px);
            width: 118px;
            height: 102px;
            clip-path: polygon(25% 0,75% 0,100% 50%,75% 100%,25% 100%,0 50%);
            border: 1px solid #1f2b43;
            background: rgba(8,13,27,.18);
            box-shadow: inset 0 0 0 2px rgba(31,43,67,.6);
        }
        .hive-node {
            position: absolute;
            left: 45%;
            top: 61%;
            width: 118px;
            height: 102px;
            transform: translate(-50%, -50%);
            display: grid;
            place-items: center;
            clip-path: polygon(25% 0,75% 0,100% 50%,75% 100%,25% 100%,0 50%);
            background: var(--h-green);
            border: 5px solid var(--h-honey);
            color: #04111a;
            box-shadow: inset 0 0 0 4px rgba(217,249,157,.85);
        }
        .hive-node strong { font-size: 30px; line-height: 1; }
        .hive-node small { margin-top: -34px; font-weight: 900; font-size: 15px; }
        .hive-card dl { display: grid; grid-template-columns: 100px 1fr; gap: 9px 12px; margin: 0; font-size: 15px; }
        .hive-card dt { color: var(--h-muted); font-weight: 900; }
        .hive-card dd { margin: 0; }
        .hive-section-title { display: flex; justify-content: space-between; align-items: end; margin-bottom: 10px; }
        .hive-section-title span { color: var(--h-muted); font-size: 12px; }
        .detections-card article {
            display: grid;
            grid-template-columns: 66px 1fr;
            grid-template-rows: auto auto;
            gap: 2px 10px;
            align-items: center;
            padding: 9px;
            border: 1px solid rgba(64,80,109,.55);
            background: rgba(11,18,32,.8);
            margin-bottom: 8px;
        }
        .detections-card article strong { font-size: 18px; }
        .detections-card article span { color: var(--h-muted); font-size: 13px; }
        .face-thumb { grid-row: 1 / 3; width: 66px; height: 48px; background: linear-gradient(135deg,#95b9dc,#1b1f28 58%,#0b0f16); }
        .face-thumb.alt { background: linear-gradient(135deg,#20242a,#86add3 62%,#111827); }
        footer { display: none !important; }
        @media (max-width: 760px) {
            .hero-shell { padding: 28px; min-height: 500px; align-items: flex-start; }
            .hero-copy h1 { font-size: 30px; }
            .tagline { font-size: 18px; }
            .complex-grid { grid-template-columns: 1fr; }
            .hive-copy { min-height: 1100px; height: auto; overflow: auto; }
            .hive-topbar, .hive-status { display: block; height: auto; }
            .hive-grid { display: block; height: auto; }
            .hive-map { height: 520px; }
        }
        .hive-live {
            height: auto;
            min-height: 0;
            overflow: visible;
            border-bottom: 0;
            border-radius: 8px 8px 0 0;
        }
        .hive-live-grid {
            display: grid !important;
            grid-template-columns: 320px minmax(0, 1fr) 360px;
            gap: 14px;
            padding: 14px;
            border: 1px solid rgba(141,160,190,.22);
            border-top: 0;
            border-radius: 0 0 8px 8px;
            background: #090d16;
        }
        .hive-live-grid > div {
            min-width: 0;
        }
        .hive-live-grid .gr-form,
        .hive-live-grid .block {
            border-color: rgba(141,160,190,.22) !important;
            background: rgba(16,24,39,.82) !important;
            border-radius: 0 !important;
        }
        .hive-live-grid label,
        .hive-live-grid .wrap label span {
            color: #9fb0c9 !important;
            font-weight: 800 !important;
        }
        .hive-live-grid button {
            min-height: 44px !important;
            border: 1px solid #40506d !important;
            border-radius: 0 !important;
            color: #edf5ff !important;
            background: #172237 !important;
            font-weight: 900 !important;
        }
        .hive-live-grid button.primary {
            color: #111827 !important;
            background: #ffb000 !important;
            border-color: #ffd057 !important;
        }
        .hive-side-title {
            color: #ffd057;
            font-size: 18px;
            font-weight: 900;
            margin: 10px 0 8px;
        }
        .hive-map-live {
            position: relative;
            min-height: 620px;
            overflow: hidden;
            border: 1px solid rgba(141,160,190,.22);
            background: #080d1b;
        }
        .hive-tool {
            min-width: 46px;
            min-height: 44px;
            display: inline-grid;
            place-items: center;
            border: 1px solid #40506d;
            background: #172237;
            color: #edf5ff;
            font-weight: 900;
        }
        .hive-tool.wide {
            padding: 0 12px;
        }
        .hive-link {
            position: absolute;
            height: 3px;
            transform-origin: 0 50%;
            border-top: 3px dashed rgba(34,197,94,.8);
            z-index: 1;
        }
        .hive-node.selected {
            box-shadow: 0 0 0 5px rgba(56,189,248,.35), inset 0 0 0 4px rgba(217,249,157,.85);
        }
        @media (max-width: 980px) {
            .hive-live-grid { display: block !important; }
            .hive-map-live { min-height: 520px; }
        }
        """,
    ) as demo:
        page_state = gr.State("simple")
        complex_state = gr.State(_complex_initial_state())
        with gr.Column(elem_classes=["wrap"]):
            language = gr.Dropdown(
                choices=list(LANGUAGES.keys()),
                value=initial_language,
                label=initial_text["language"],
                elem_classes=["lang-select"],
            )
            with gr.Group(visible=False) as landing_group:
                intro_html = gr.HTML(render_intro(initial_language))
                with gr.Row(elem_classes=["choice-row"]):
                    simple_button = gr.Button(initial_text["simple"], variant="primary")
                    complex_button = gr.Button(initial_text["complex"], variant="secondary", link="/hive-web", link_target="_self")
            with gr.Group(visible=True) as simple_group:
                simple_header = gr.HTML(render_simple_header(initial_language))
                with gr.Row(elem_classes=["back-row"]):
                    back_from_simple = gr.Button(initial_text["back"], variant="secondary")
                with gr.Row():
                    with gr.Column(scale=1):
                        image = gr.Image(type="filepath", label=initial_text["image"])
                        mode = gr.Radio(["GPU", "CPU"], value="GPU", label=initial_text["mode"])
                        min_score = gr.Slider(0.70, 0.99, value=float(active.min_score), step=0.01, label=initial_text["score"])
                        min_margin = gr.Slider(0.00, 0.20, value=float(active.min_margin), step=0.005, label=initial_text["margin"])
                        button = gr.Button(initial_text["recognize"], variant="primary")
                    with gr.Column(scale=1):
                        summary = gr.Markdown(initial_text["summary"])
                        result_json = gr.JSON(label=initial_text["json"])
                examples = example_images()
                if examples:
                    gr.Examples(
                        examples=examples,
                        inputs=[image, mode, min_score, min_margin],
                        label=initial_text["examples"],
                    )
            with gr.Group(visible=False) as complex_group:
                complex_html = gr.HTML(render_complex_topbar(initial_language, _complex_initial_state()))
                with gr.Row(elem_classes=["hive-live-grid"]):
                    with gr.Column(scale=1):
                        gr.HTML('<div class="hive-side-title">Processors</div>')
                        hive_distance = gr.Textbox(value="1,0", label="distance")
                        hive_signal = gr.Textbox(value="1,0", label="signal")
                        with gr.Row():
                            hive_add = gr.Button("Add processor", variant="primary")
                            hive_reset = gr.Button("Reset")
                        with gr.Row():
                            hive_step = gr.Button("Step selected")
                            hive_run = gr.Button("Run selected")
                        with gr.Row():
                            hive_all_cpu = gr.Button("All CPU")
                            hive_all_gpu = gr.Button("All GPU")
                        gr.HTML('<div class="hive-side-title">Detector</div>')
                        hive_image = gr.Image(type="filepath", label="Manual photo")
                        hive_min_score = gr.Slider(0.70, 0.99, value=float(active.min_score), step=0.01, label="min score")
                        hive_min_margin = gr.Slider(0.00, 0.20, value=float(active.min_margin), step=0.005, label="min margin")
                        gr.HTML('<div class="hive-side-title">Network</div>')
                        hive_managers = gr.Textbox(value="2", label="managers")
                        with gr.Row():
                            hive_hierarchy = gr.Button("Build hierarchy")
                            hive_random = gr.Button("Randomize links")
                        with gr.Row():
                            hive_matrix = gr.Button("Matrix plan")
                            hive_fit = gr.Button("Fit map")
                        gr.HTML('<div class="hive-side-title">Program</div>')
                        with gr.Row():
                            hive_default = gr.Button("Default")
                            hive_controller = gr.Button("Controller")
                    with gr.Column(scale=2):
                        complex_map = gr.HTML(render_complex_map(_complex_initial_state()))
                        complex_status = gr.Markdown("Upload an image and press GPU/CPU to run recognition.")
                        complex_payload = gr.JSON(label="Detector result")
                    with gr.Column(scale=1):
                        complex_details = gr.HTML(render_complex_details(_complex_initial_state()))
                        complex_detections = gr.HTML(render_complex_detections(_complex_initial_state()))
                with gr.Row(elem_classes=["back-row"]):
                    back_from_complex = gr.Button(initial_text["back"], variant="secondary")

        navigation_outputs = [
            page_state,
            landing_group,
            simple_group,
            complex_group,
            intro_html,
            simple_header,
            complex_html,
            simple_button,
            complex_button,
            back_from_simple,
            back_from_complex,
            image,
            mode,
            min_score,
            min_margin,
            button,
            result_json,
            summary,
        ]
        language.change(lambda selected, page: navigate(selected, page), [language, page_state], navigation_outputs)
        simple_button.click(lambda selected: navigate(selected, "simple"), language, navigation_outputs)
        # The complex demo button opens the one-to-one Hive Web copy directly.
        back_from_simple.click(lambda selected: navigate(selected, "landing"), language, navigation_outputs)
        back_from_complex.click(lambda selected: navigate(selected, "landing"), language, navigation_outputs)
        button.click(detect_uploaded, [image, mode, min_score, min_margin], [summary, result_json], api_name="recognize")
        hive_mutation_outputs = [complex_state, complex_html, complex_map, complex_details, complex_detections]
        for hive_button, hive_action in [
            (hive_add, "add"),
            (hive_reset, "reset"),
            (hive_step, "step"),
            (hive_hierarchy, "hierarchy"),
            (hive_random, "randomize"),
            (hive_fit, "randomize"),
            (hive_matrix, "hierarchy"),
            (hive_default, "default"),
            (hive_controller, "controller"),
        ]:
            hive_button.click(
                lambda selected_language, state, action=hive_action: _complex_mutate(selected_language, state, action),
                [language, complex_state],
                hive_mutation_outputs,
            )
        hive_run.click(
            lambda image_path, state, selected_language, score, margin: _complex_detect(image_path, state, selected_language, "GPU", score, margin),
            [hive_image, complex_state, language, hive_min_score, hive_min_margin],
            [complex_state, complex_html, complex_map, complex_details, complex_detections, complex_status, complex_payload],
        )
        hive_all_gpu.click(
            lambda image_path, state, selected_language, score, margin: _complex_detect(image_path, state, selected_language, "GPU", score, margin),
            [hive_image, complex_state, language, hive_min_score, hive_min_margin],
            [complex_state, complex_html, complex_map, complex_details, complex_detections, complex_status, complex_payload],
        )
        hive_all_cpu.click(
            lambda image_path, state, selected_language, score, margin: _complex_detect(image_path, state, selected_language, "CPU", score, margin),
            [hive_image, complex_state, language, hive_min_score, hive_min_margin],
            [complex_state, complex_html, complex_map, complex_details, complex_detections, complex_status, complex_payload],
        )
    return demo


def run_selftest() -> list[dict]:
    active = load_detector()
    rows: list[dict] = []
    for label, mode, min_score, min_margin in example_images():
        _summary_text, payload = detect_uploaded(label, mode, min_score, min_margin)
        rows.append(
            {
                "image": label,
                "accepted": payload.get("accepted"),
                "identity": payload.get("identity"),
                "best_label": payload.get("best_label"),
                "best_score": payload.get("best_score"),
                "margin": payload.get("margin"),
            }
        )
    return rows


def main() -> str:
    demo = build_demo()
    demo.queue(default_concurrency_limit=4)
    app, local_url, share_url = demo.launch(
        share=SHARE,
        server_name="0.0.0.0" if SHARE else "127.0.0.1",
        server_port=PORT,
        prevent_thread_lock=True,
        quiet=False,
    )
    register_hive_web_routes(app)
    public_url = (share_url or local_url).rstrip("/")
    URL_FILE.parent.mkdir(parents=True, exist_ok=True)
    URL_FILE.write_text(public_url, encoding="utf-8")
    print("PUBLIC_SITE_URL=" + public_url, flush=True)
    print("Programmatic endpoint: /gradio_api/call/recognize", flush=True)
    print(json.dumps({"selftest": run_selftest()}, indent=2), flush=True)
    if BLOCK:
        try:
            while True:
                time.sleep(60)
        except KeyboardInterrupt:
            try:
                demo.close()
            except Exception:
                pass
    return public_url


if __name__ == "__main__":
    main()
